In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv('/content/resume_dataset_200k_enhanced[1].csv')

In [ ]:
df.isnull().sum()

,0
candidate_id,0
age,0
education_level,0
university_tier,0
cgpa,0
internships,0
projects,0
programming_languages,0
certifications,0
experience_years,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df.drop('hired', axis=1)
y = df['hired']

categorical_features = ['education_level', 'university_tier', 'company_type']
numerical_features = ['age', 'cgpa', 'internships', 'projects', 'experience_years', 'hackathons', 'research_papers', 'skills_score', 'soft_skills_score', 'resume_length_words']

X['programming_languages_count'] = X['programming_languages'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() != '' else 0)
X['certifications_count'] = X['certifications'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() != '' else 0)

numerical_features.extend(['programming_languages_count', 'certifications_count'])
X = X.drop(['programming_languages', 'certifications'], axis=1)

X = X.drop('candidate_id', axis=1)

print(f"Original numerical features: {numerical_features}")
print(f"Original categorical features: {categorical_features}")
print(f"Updated X columns after feature engineering: {X.columns.tolist()}")


Original numerical features: ['age', 'cgpa', 'internships', 'projects', 'experience_years', 'hackathons', 'research_papers', 'skills_score', 'soft_skills_score', 'resume_length_words', 'programming_languages_count', 'certifications_count']
Original categorical features: ['education_level', 'university_tier', 'company_type']
Updated X columns after feature engineering: ['age', 'education_level', 'university_tier', 'cgpa', 'internships', 'projects', 'experience_years', 'hackathons', 'research_papers', 'skills_score', 'soft_skills_score', 'resume_length_words', 'company_type', 'programming_languages_count', 'certifications_count']


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (160000, 15)
X_test shape: (40000, 15)
y_train shape: (160000,)
y_test shape: (40000,)


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', SVC(kernel='linear', random_state=42))])

print("Training SVM model...")
svm_pipeline.fit(X_train, y_train)
print("SVM model training complete.")

Training SVM model...
SVM model training complete.


In [ ]:
y_pred = svm_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7060


In [ ]:
import xgboost as xgb
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import os # Import the os module

# Define the file path
file_path = '/content/resume_dataset_200k_enhanced[1].csv'

# Check if the file exists before attempting to load it
if not os.path.exists(file_path):
    print(f"Error: The file '{file_path}' was not found.")
    print("Please ensure the CSV file is uploaded to the Colab environment.")
    print("You might need to re-run the `files.upload()` cell or check the file path.")
else:
    # Load the DataFrame (re-added for robustness)
    df = pd.read_csv(file_path)

    # Separate features (X) and target (y)
    X = df.drop('hired', axis=1)
    y = df['hired']

    # Redefine numerical and categorical features (as they were after initial feature engineering)
    categorical_features = ['education_level', 'university_tier', 'company_type']
    numerical_features = ['age', 'cgpa', 'internships', 'projects', 'experience_years', 'hackathons', 'research_papers', 'skills_score', 'soft_skills_score', 'resume_length_words']

    # For 'programming_languages' and 'certifications', convert them to numerical counts
    X['programming_languages_count'] = X['programming_languages'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() != '' else 0)
    X['certifications_count'] = X['certifications'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() != '' else 0)

    # Add these new count columns to numerical_features and remove original string columns
    numerical_features.extend(['programming_languages_count', 'certifications_count'])
    X = X.drop(['programming_languages', 'certifications'], axis=1)

    # Remove 'candidate_id' as it's just an identifier
    X = X.drop('candidate_id', axis=1)

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Redefine preprocessor (uses the now correctly defined numerical_features and categorical_features)
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])

    xgb_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('classifier', xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42))])

    print("Training XGBoost model...")
    xgb_pipeline.fit(X_train, y_train)
    print("XGBoost model training complete.")

Training XGBoost model...
XGBoost model training complete.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report # Added import

y_pred_xgb = xgb_pipeline.predict(X_test)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
report_xgb = classification_report(y_test, y_pred_xgb)

print(f"XGBoost Accuracy: {accuracy_xgb:.4f}")
print("XGBoost Classification Report:\n", report_xgb)

XGBoost Accuracy: 0.7035
XGBoost Classification Report:
               precision    recall  f1-score   support

           0       0.29      0.01      0.01     11758
           1       0.71      0.99      0.83     28242

    accuracy                           0.70     40000
   macro avg       0.50      0.50      0.42     40000
weighted avg       0.59      0.70      0.59     40000



In [ ]:
print(df['hired'].value_counts())
print(df['hired'].value_counts(normalize=True))

hired
1    141212
0     58788
Name: count, dtype: int64
hired
1    0.70606
0    0.29394
Name: proportion, dtype: float64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

[[   73 11685]
 [  175 28067]]
              precision    recall  f1-score   support

           0       0.29      0.01      0.01     11758
           1       0.71      0.99      0.83     28242

    accuracy                           0.70     40000
   macro avg       0.50      0.50      0.42     40000
weighted avg       0.59      0.70      0.59     40000



In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd # Import pandas to convert sparse matrix back to DataFrame

smo = SMOTE(random_state=42)

# Transform X_train using the preprocessor first
X_train_processed = preprocessor.fit_transform(X_train)

# SMOTE works on the numerical data, which can be a sparse matrix from OneHotEncoder
X_train_resampled, y_train_resampled = smo.fit_resample(X_train_processed, y_train)

# If you need X_train_resampled back as a DataFrame with feature names (optional, but good for inspection):
# Get feature names after one-hot encoding
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + list(cat_feature_names)

# Convert the resampled data back to a DataFrame
X_train = pd.DataFrame(X_train_resampled, columns=all_feature_names)
y_train = y_train_resampled

print(f"Original X_train shape: {X_train_processed.shape}")
print(f"Resampled X_train shape: {X_train.shape}")
print(f"Resampled y_train shape: {y_train.shape}")
print(f"Value counts of y_train after SMOTE:\n{y_train.value_counts()}")

Original X_train shape: (160000, 21)
Resampled X_train shape: (225940, 21)
Resampled y_train shape: (225940,)
Value counts of y_train after SMOTE:
hired
1    112970
0    112970
Name: count, dtype: int64


In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Extract the XGBoost classifier from the pipeline for direct fitting on processed data
xgb_classifier_resampled = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

print("Retraining XGBoost model with SMOTE-resampled data...")
# Fit the classifier directly on the already preprocessed and resampled X_train
xgb_classifier_resampled.fit(X_train, y_train)
print("XGBoost model retraining complete with SMOTE data.")

# Transform X_test using the preprocessor that was fitted on the original X_train
X_test_processed = preprocessor.transform(X_test)

# Make predictions on the transformed X_test
y_pred_xgb_resampled = xgb_classifier_resampled.predict(X_test_processed)

# Evaluate the model
accuracy_xgb_resampled = accuracy_score(y_test, y_pred_xgb_resampled)
report_xgb_resampled = classification_report(y_test, y_pred_xgb_resampled)
confusion_xgb_resampled = confusion_matrix(y_test, y_pred_xgb_resampled)

print(f"\nXGBoost (SMOTE) Accuracy: {accuracy_xgb_resampled:.4f}")
print("\nXGBoost (SMOTE) Classification Report:\n", report_xgb_resampled)
print("\nXGBoost (SMOTE) Confusion Matrix:\n", confusion_xgb_resampled)

Retraining XGBoost model with SMOTE-resampled data...
XGBoost model retraining complete with SMOTE data.

XGBoost (SMOTE) Accuracy: 0.7042

XGBoost (SMOTE) Classification Report:
               precision    recall  f1-score   support

           0       0.33      0.01      0.01     11758
           1       0.71      0.99      0.83     28242

    accuracy                           0.70     40000
   macro avg       0.52      0.50      0.42     40000
weighted avg       0.60      0.70      0.59     40000


XGBoost (SMOTE) Confusion Matrix:
 [[   74 11684]
 [  147 28095]]


Let's demonstrate how to use the retrained XGBoost model (with SMOTE) to predict whether a sample candidate would be hired.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from google.colab import files
import os
import xgboost as xgb
from imblearn.over_sampling import SMOTE

file_path = '/content/resume_dataset_200k_enhanced[1].csv'

if not os.path.exists(file_path):
    print(f"File '{file_path}' not found. Please upload the file.")
    uploaded = files.upload()

df = pd.read_csv(file_path)

X = df.drop('hired', axis=1)
y = df['hired']

numerical_features = ['age', 'cgpa', 'internships', 'projects', 'experience_years', 'hackathons', 'research_papers', 'skills_score', 'soft_skills_score', 'resume_length_words']
categorical_features = ['education_level', 'university_tier', 'company_type']

X['programming_languages_count'] = X['programming_languages'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() != '' else 0)
X['certifications_count'] = X['certifications'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() != '' else 0)

numerical_features.extend(['programming_languages_count', 'certifications_count'])
X = X.drop(['programming_languages', 'certifications'], axis=1)
X = X.drop('candidate_id', axis=1)

_X_train_original, _X_test_original, _y_train_original, _y_test_original = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

preprocessor.fit(_X_train_original)

_train_processed = preprocessor.transform(_X_train_original)
X_train, y_train = smo.fit_resample(X_train_processed, _y_train_original)

xgb_classifier_resampled = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)
print("Retraining XGBoost model with SMOTE-resampled data for prediction...")
xgb_classifier_resampled.fit(X_train, y_train)
print("XGBoost model retraining complete.")



sample_candidate = pd.DataFrame({
    'age': [25],
    'education_level': ['Bachelors'],
    'university_tier': ['Tier 2'],
    'cgpa': [8.5],
    'internships': [2],
    'projects': [3],
    'experience_years': [2],
    'hackathons': [1],
    'research_papers': [0],
    'skills_score': [75],
    'soft_skills_score': [80],
    'resume_length_words': [300],
    'company_type': ['Startup'],
    'programming_languages_count': [3],
    'certifications_count': [1]
})

print("\nSample Candidate Data:")
display(sample_candidate)

sample_candidate_processed = preprocessor.transform(sample_candidate)

prediction = xgb_classifier_resampled.predict(sample_candidate_processed)

if prediction[0] == 1:
    print("\nPrediction: This candidate is likely to be Hired.")
else:
    print("\nPrediction: This candidate is likely to be Not Hired.")

Retraining XGBoost model with SMOTE-resampled data for prediction...
XGBoost model retraining complete.

Sample Candidate Data:


,age,education_level,university_tier,cgpa,internships,projects,experience_years,hackathons,research_papers,skills_score,soft_skills_score,resume_length_words,company_type,programming_languages_count,certifications_count
0,25,Bachelors,Tier 2,8.5,2,3,2,1,0,75,80,300,Startup,3,1



Prediction: This candidate is likely to be Hired.
